# CS4248 — Sentiment Analysis (LR / NB-weighted LR)

Run each section in order. GPU is **not** needed — these are CPU-only models.

## 1. Clone repo & install dependencies

In [44]:
import os

REPO_URL    = "https://github.com/Mohammed-Faizzzz/CS4248_PROJ.git"
BRANCH      = "faiz-lr"   # change to your branch if needed
REPO_DIR    = "CS4248_PROJ"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

Cloning into 'CS4248_PROJ'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 91 (delta 31), reused 79 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 1.53 MiB | 5.21 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/CS4248_PROJ/CS4248_PROJ/CS4248_PROJ/CS4248_PROJ


In [45]:
# Install dependencies from pyproject.toml
!pip install -q nltk pandas scikit-learn scipy
import pandas as pd
import numpy as np

import nltk
for pkg in ["stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
            "punkt", "punkt_tab"]:
    nltk.download(pkg, quiet=True)

print("Dependencies ready.")

Dependencies ready.


## 2. Mount Google Drive and set data paths

Upload `train.csv` and `test.csv` from the Kaggle Tweet Sentiment dataset to your Drive,
then set the paths below. Alternatively, skip Drive and use the file upload widget.

In [47]:
USE_DRIVE = True  # set False to upload files manually instead

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    # ── Edit these paths to match where you put the files in your Drive ──
    DRIVE_TRAIN = "/content/drive/MyDrive/CS4248/train.csv"
    DRIVE_TEST  = "/content/drive/MyDrive/CS4248/test.csv"
    ELON_TWEETS = "/content/drive/MyDrive/CS4248/elon_tweets.csv"
    # ─────────────────────────────────────────────────────────────────────

    import shutil, os
    os.makedirs("data/raw", exist_ok=True)
    shutil.copy(DRIVE_TRAIN, "data/raw/train.csv")
    shutil.copy(DRIVE_TEST,  "data/raw/test.csv")
    shutil.copy(ELON_TWEETS, "data/raw/elon_tweets.csv")
    print("Copied data from Drive.")

else:
    # Manual upload fallback
    from google.colab import files
    import os
    os.makedirs("data/raw", exist_ok=True)
    print("Select train.csv then test.csv when prompted.")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        with open(f"data/raw/{fname}", "wb") as f:
            f.write(data)
    print("Uploaded:", list(uploaded.keys()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied data from Drive.


## 3. Prepare splits

Cleans raw text and writes `data/splits/train.csv` and `data/splits/test.csv`.

In [38]:
!python scripts/prepare_splits.py \
    --train-path data/raw/train.csv \
    --test-path  data/raw/test.csv \
    --output-dir data/splits

Loading and cleaning train set...

Train class distribution:
    negative   7,781  (28.3%)
     neutral  11,116  (40.5%)
    positive   8,582  (31.2%)
       total  27,479

Loading and cleaning test set...

Test class distribution:
    negative   1,001  (28.3%)
     neutral   1,430  (40.5%)
    positive   1,103  (31.2%)
       total   3,534

Saved -> data/splits/train.csv
Saved -> data/splits/test.csv


## 4a. Train — plain Logistic Regression (baseline)

In [39]:
!python -m scripts.lr.train \
    --train data/splits/train.csv \
    --test  data/splits/test.csv \
    --save-model models/lr_baseline.pkl

  LR Experiment Config
  NB-weighted:     False
  Stemming:        False
  Lemmatization:   False
  Bigrams:         False
  Trigrams:        False
  Char n-grams:    False
  Negation:        True
  Penalty:         l2
  C:               1.0
  Class weight:    None
  Drop neutral:    False
  Downsample:      False
  Optimise:        False

Loading data...
Preprocessing text...
Train: 27,479  Test: 3,534
Building TF-IDF features...
Feature matrix: 27,479 rows × 10,619 features

LogisticRegression config: {'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 2000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'liblinear', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
Running 5-fold CV...
Macro F1 (CV): 0.6923 ± 0.0028
              precision    recall  f1-score   support

    negative       0.74      0.63      0.68      1001
     neutral       0.64      0.76      0.70 

## 4b. Train — NB-weighted LR (Wang & Manning 2012)

In [40]:
!python -m scripts.lr.train \
    --train data/splits/train.csv \
    --test  data/splits/test.csv \
    --nb-weighted \
    --nb-alpha 1.0 \
    --C 5.0 \
    --use-gpu \
    --use-char-ngrams \
    --save-model models/nb_lr.pkl

  NB-weighted LR Experiment Config
  NB-weighted:     True
  NB alpha:        1.0
  GPU (cuML):      True
  Stemming:        False
  Lemmatization:   False
  Bigrams:         False
  Trigrams:        False
  Char n-grams:    True
  Negation:        True
  Penalty:         l2
  C:               5.0
  Class weight:    None
  Drop neutral:    False
  Downsample:      False
  Optimise:        False

Loading data...
Preprocessing text...
Train: 27,479  Test: 3,534
Building TF-IDF features...
Feature matrix: 27,479 rows × 91,400 features

NBWeightedLR config: {'C': 5.0, 'alpha': 1.0, 'class_weight': None, 'l1_ratio': 0.0, 'use_gpu': True}
Running 5-fold CV...
Macro F1 (CV): 0.7236 ± 0.0036
              precision    recall  f1-score   support

    negative       0.74      0.69      0.72      1001
     neutral       0.68      0.75      0.71      1430
    positive       0.82      0.76      0.79      1103

    accuracy                           0.74      3534
   macro avg       0.75      0.73  

## 4c. Hyperparameter search — NB-weighted LR

Searches over `C ∈ {0.1, 1, 5, 10}`, `alpha ∈ {0.1, 0.5, 1, 2}`, `l1_ratio`, and `class_weight`.
Takes ~10–20 min depending on dataset size.

In [41]:
!python -m scripts.lr.train \
    --train data/splits/train.csv \
    --test  data/splits/test.csv \
    --nb-weighted \
    --use-gpu \
    --optimise \
    --save-model models/nb_lr_optimised.pkl

  NB-weighted LR Experiment Config
  NB-weighted:     True
  NB alpha:        1.0
  GPU (cuML):      True
  Stemming:        False
  Lemmatization:   False
  Bigrams:         False
  Trigrams:        False
  Char n-grams:    False
  Negation:        True
  Penalty:         l2
  C:               1.0
  Class weight:    None
  Drop neutral:    False
  Downsample:      False
  Optimise:        True

Loading data...
Preprocessing text...
Train: 27,479  Test: 3,534
Building TF-IDF features...
Feature matrix: 27,479 rows × 10,619 features

Running RandomizedSearchCV (NB-weighted LR, 30 iterations, GPU=True)...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best params:   C=1.1007, alpha=0.5, l1_ratio=0.0, class_weight=balanced
Best CV macro F1: 0.7219

--- Optimised model on held-out test set ---
              precision    recall  f1-score   support

    negative       0.72      0.69      0.70      1001
     neutral       0.67      0.73      0.70      1430
    positive       0

In [42]:
!python -m scripts.lr.train \
    --train data/splits/train.csv \
    --test  data/splits/test.csv \
    --nb-weighted \
    --optimise \
    --use-trigrams \
    --use-char-ngrams \
    --use-gpu \
    --save-model models/nb_lr_optimised.pkl

  NB-weighted LR Experiment Config
  NB-weighted:     True
  NB alpha:        1.0
  GPU (cuML):      True
  Stemming:        False
  Lemmatization:   False
  Bigrams:         False
  Trigrams:        True
  Char n-grams:    True
  Negation:        True
  Penalty:         l2
  C:               1.0
  Class weight:    None
  Drop neutral:    False
  Downsample:      False
  Optimise:        True

Loading data...
Preprocessing text...
Train: 27,479  Test: 3,534
Building TF-IDF features...
Feature matrix: 27,479 rows × 135,485 features

Running RandomizedSearchCV (NB-weighted LR, 30 iterations, GPU=True)...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best params:   C=3.7184, alpha=1.0, l1_ratio=0.0, class_weight=balanced
Best CV macro F1: 0.7354

--- Optimised model on held-out test set ---
              precision    recall  f1-score   support

    negative       0.74      0.73      0.73      1001
     neutral       0.70      0.72      0.71      1430
    positive       0.

In [43]:
!python -m scripts.lr.train \
    --train data/splits/train.csv \
    --test  data/splits/test.csv \
    --nb-weighted \
    --class-weight balanced \
    --optimise \
    --use-trigrams \
    --use-char-ngrams \
    --use-gpu \
    --save-model models/nb_lr_optimised.pkl

  NB-weighted LR Experiment Config
  NB-weighted:     True
  NB alpha:        1.0
  GPU (cuML):      True
  Stemming:        False
  Lemmatization:   False
  Bigrams:         False
  Trigrams:        True
  Char n-grams:    True
  Negation:        True
  Penalty:         l2
  C:               1.0
  Class weight:    balanced
  Drop neutral:    False
  Downsample:      False
  Optimise:        True

Loading data...
Preprocessing text...
Train: 27,479  Test: 3,534
Building TF-IDF features...
Feature matrix: 27,479 rows × 135,485 features

Running RandomizedSearchCV (NB-weighted LR, 30 iterations, GPU=True)...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best params:   C=6.9783, alpha=2.0, l1_ratio=0.0, class_weight=balanced
Best CV macro F1: 0.7351

--- Optimised model on held-out test set ---
              precision    recall  f1-score   support

    negative       0.73      0.74      0.73      1001
     neutral       0.70      0.71      0.71      1430
    positive     

## 5. (Optional) Save trained model back to Drive

In [ ]:
import shutil, os

SAVE_TO_DRIVE = True
DRIVE_OUT_DIR = "/content/drive/MyDrive/CS4248/models"

if SAVE_TO_DRIVE:
    os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
    for pkl in ["models/lr_baseline.pkl", "models/nb_lr.pkl", "models/nb_lr_optimised.pkl"]:
        if os.path.exists(pkl):
            shutil.copy(pkl, DRIVE_OUT_DIR)
            print(f"Saved {pkl} → {DRIVE_OUT_DIR}")